In [22]:
from sqlalchemy import Column, Integer, String, ForeignKey, Table, create_engine
from sqlalchemy.orm import relationship, sessionmaker, declarative_base

# Configuration - Database for Hollywood task
engine_movies = create_engine('sqlite:///movies.db', echo=False)
Base = declarative_base()

In [23]:
# The "Bridge" table (Many-to-Many association)
movie_actor_association = Table(
    'movie_actor', 
    Base.metadata,
    Column('movie_id', Integer, ForeignKey('movies.id')),
    Column('actor_id', Integer, ForeignKey('actors.id'))
)

In [24]:
class Movie(Base):
    __tablename__ = 'movies'
    id = Column(Integer, primary_key=True)
    title = Column(String(200), nullable=False)
    # Linking movies to actors via the secondary table
    actors = relationship("Actor", secondary=movie_actor_association, back_populates="movies")

class Actor(Base):
    __tablename__ = 'actors'
    id = Column(Integer, primary_key=True)
    name = Column(String(100), nullable=False)
    # Linking actors back to movies
    movies = relationship("Movie", secondary=movie_actor_association, back_populates="actors")

In [25]:
# Create physical tables in movies.db
Base.metadata.create_all(engine_movies)

# Start a session
Session = sessionmaker(bind=engine_movies)
session = Session()

In [26]:
try:
    # 1. Create instances
    m1 = Movie(title="Inception")
    m2 = Movie(title="The Revenant")
    a1 = Actor(name="Leonardo DiCaprio")
    a2 = Actor(name="Tom Hardy")

    # 2. Establish connections (Many-to-Many)
    m1.actors = [a1, a2]
    m2.actors = [a1, a2]

    # 3. Save to database
    session.add_all([m1, m2, a1, a2])
    session.commit()
    
    print("✅ Success! Movies and Actors are linked in the database.")
    
    # 4. Quick check
    test_movie = session.query(Movie).first()
    print(f"Verified: {test_movie.title} cast -> {[a.name for a in test_movie.actors]}")

except Exception as e:
    session.rollback()
    print(f"❌ Could not save data: {e}")

✅ Success! Movies and Actors are linked in the database.
Verified: Inception cast -> ['Leonardo DiCaprio', 'Tom Hardy']
